# Notebook 1 — API Connection

## Project: Market Expansion Strategy — Travel Company Analysis

### Business Context
This project was built for a global online travel company competing with Booking Holdings, Expedia, and Airbnb. The company needs a data-driven framework to decide **which 5 countries deserve the most marketing investment**.

To answer this question, we retrieve data from two external sources:
- **World Bank API** — provides country-level economic and infrastructure indicators
- **Google Trends (PyTrends)** — provides country-level online travel search interest

### Purpose of This Notebook
This notebook proves that both API connections work correctly before any data is pulled or analyzed. It also retrieves the full World Bank indicator catalog and saves it as a CSV file so it can be reused in later notebooks without repeated API calls.

In [1]:
# Importing libraries

import numpy as np
import pandas as pd

In [2]:
# requests calls the API.
# time helps us pause between API calls so we do not overload the server.

import requests
import time

## API 1: World Bank

- API1 is World Bank. World Bank (https://documents.worldbank.org/en/publication/documents-reports/api) has multiple APIs.
- For our project, we need World Bank Indicators API which includes GDP Growth, GDP per Capita, internet users %, population information for countries.
- API Document website explains how to

In [3]:
# Connecting to the World Bank Indicators API endpoint

url = "https://api.worldbank.org/v2/indicator"

response = requests.get(url)
print(response.status_code)

200


In [4]:
# Checking the type of result

response.text[:1000]

'ï»¿<?xml version="1.0" encoding="utf-8"?>\r\n<wb:indicators page="1" pages="591" per_page="50" total="29512" xmlns:wb="http://www.worldbank.org">\r\n  <wb:indicator id="1.0.HCount.1.90usd">\r\n    <wb:name>Poverty Headcount ($1.90 a day)</wb:name>\r\n    <wb:unit />\r\n    <wb:source id="37">LAC Equity Lab</wb:source>\r\n    <wb:sourceNote>The poverty headcount index measures the proportion of the population with daily per capita income (in 2011 PPP) below the poverty line.</wb:sourceNote>\r\n    <wb:sourceOrganization>LAC Equity Lab tabulations of SEDLAC (CEDLAS and the World Bank).</wb:sourceOrganization>\r\n    <wb:topics>\r\n      <wb:topic id="11">Poverty </wb:topic>\r\n    </wb:topics>\r\n  </wb:indicator>\r\n  <wb:indicator id="1.0.HCount.2.5usd">\r\n    <wb:name>Poverty Headcount ($2.50 a day)</wb:name>\r\n    <wb:unit />\r\n    <wb:source id="37">LAC Equity Lab</wb:source>\r\n    <wb:sourceNote>The poverty headcount index measures the proportion of the population with daily p

In [5]:
# Turning the result into JSON since we can't work with XML

url = "https://api.worldbank.org/v2/indicator"

params = {
    "format": "json"
}

response = requests.get(url, params=params)

print(response.status_code)

200


In [6]:
# Checking the type of the data in JSON

world_bank_data = response.json()

type(world_bank_data)

list

In [7]:
# Checking how many elements exist in the top-level list

len(world_bank_data)

2

In [8]:
# Displaying the metadata section

world_bank_data [0]

{'page': 1, 'pages': 591, 'per_page': '50', 'total': 29512}

In [9]:
# This tells us that the World Bank API contains 29,512 indicators spread across 591 pages.
# Since only 50 indicators are returned per page by default, we will need pagination to retrieve the complete indicator catalog.

In [10]:
# Displaying the first indicator record

world_bank_data [1][0]

{'id': '1.0.HCount.1.90usd',
 'name': 'Poverty Headcount ($1.90 a day)',
 'unit': '',
 'source': {'id': '37', 'value': 'LAC Equity Lab'},
 'sourceNote': 'The poverty headcount index measures the proportion of the population with daily per capita income (in 2011 PPP) below the poverty line.',
 'sourceOrganization': 'LAC Equity Lab tabulations of SEDLAC (CEDLAS and the World Bank).',
 'topics': [{'id': '11', 'value': 'Poverty '}]}

In [11]:
# Listing all keys available in one indicator record

world_bank_data [1][0] . keys ()

dict_keys(['id', 'name', 'unit', 'source', 'sourceNote', 'sourceOrganization', 'topics'])

In [12]:
# To get all indicators in one calling and 1 page and not dealing with pagination, we will adjust the initial API calling code.

url = "https://api.worldbank.org/v2/indicator"

params = {
    "format": "json",
    "per_page": 29512
}

response = requests.get(url, params=params)

world_bank_data = response.json()

world_bank_data[0]

{'page': 1, 'pages': 1, 'per_page': '29512', 'total': 29512}

In [13]:
# Extracting the actual indicator records from the API response
all_indicators = world_bank_data[1]

# Creating a dataframe from all World Bank indicators
indicator_catalog_df = pd.DataFrame(all_indicators)

indicator_catalog_df.head()

,id,name,unit,source,sourceNote,sourceOrganization,topics
0,1.0.HCount.1.90usd,Poverty Headcount ($1.90 a day),,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of SEDLAC (CEDLAS a...,"[{'id': '11', 'value': 'Poverty '}]"
1,1.0.HCount.2.5usd,Poverty Headcount ($2.50 a day),,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of SEDLAC (CEDLAS a...,"[{'id': '11', 'value': 'Poverty '}]"
2,1.0.HCount.Mid10to50,Middle Class ($10-50 a day) Headcount,,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of SEDLAC (CEDLAS a...,"[{'id': '11', 'value': 'Poverty '}]"
3,1.0.HCount.Ofcl,Official Moderate Poverty Rate-National,,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of data from Nation...,"[{'id': '11', 'value': 'Poverty '}]"
4,1.0.HCount.Poor4uds,Poverty Headcount ($4 a day),,"{'id': '37', 'value': 'LAC Equity Lab'}",The poverty headcount index measures the propo...,LAC Equity Lab tabulations of SEDLAC (CEDLAS a...,"[{'id': '11', 'value': 'Poverty '}]"


In [14]:
# Saving the full World Bank indicator catalog
# This file is used in Notebook 2 to search for and confirm our KPI indicator codes
# Saving it here means we do not need to call the API again in later notebooks
indicator_catalog_df.to_csv("world_bank_indicator_catalog.csv", index=False)

## API 2: Google Trends (PyTrends)

- Google Trends provides consumer level signal of travel demand that economic indicators cannot capture. 
- A country may have strong GDP growth but low actual travel interest — or vice versa. 
- Including travel search interest makes our Market Attractiveness Framework more complete.


In [15]:
# Install pytrends for Google Trends API
%pip install pytrends

Note: you may need to restart the kernel to use updated packages.


In [16]:
# We import the tools:
    #TrendReq creates the Google Trends connection.
    #time lets us wait before sending requests.
    #sleep (10): pause for 10 seconds before sending another request to not get 429 error.

from pytrends.request import TrendReq
import time
time.sleep(10)

In [17]:
# We create a connection object.
# hl="en-US": the language is English.
# tz=360: the timezone offset used by Google Trends.

pytrends = TrendReq(hl="en-US", tz=360)

In [18]:
# Testing

keyword = ["Travel"]

pytrends.build_payload(
    kw_list=keyword,
    timeframe="today 5-y"
)

trends_df = pytrends.interest_over_time()

trends_df.head()

/opt/anaconda3/lib/python3.13/site-packages/pytrends/request.py:260: FutureWarning: Downcasting object dtype arrays on .fillna, .ffill, .bfill is deprecated and will change in a future version. Call result.infer_objects(copy=False) instead. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.fillna(False)


,Travel,isPartial
date,,
2021-06-20,47,False
2021-06-27,47,False
2021-07-04,46,False
2021-07-11,48,False
2021-07-18,49,False


In [19]:
# The Google Trends API connection was tested successfully using the keyword "Travel".
# Country-level Google Trends extraction was not finalized in this notebook because PyTrends may return request errors when repeated or country-level requests are sent.
# The final Google Trends country-level dataset will be created in the EDA notebook after the Top 30 country universe is finalized from World Bank GDP data.
# This keeps the API notebook focused on proving the connection and avoids unnecessary repeated Google Trends requests.